# Databricks Cloud Connection Notebook

This notebook demonstrates how to connect to Databricks cloud services and manage notebooks programmatically.

## Prerequisites
- Databricks workspace URL
- Databricks personal access token (PAT)
- Python 3.7+

## What you'll learn:
1. Install and configure Databricks CLI and SDK
2. Authenticate with Databricks workspace
3. Create and manage notebooks programmatically
4. Connect to cloud storage (AWS S3, Azure Blob, GCP)
5. Test cloud connectivity

## 1. Install Databricks CLI and SDK

First, install the required packages for connecting to Databricks.

In [42]:
# Install Databricks CLI and SDK
!pip install databricks-cli databricks-sdk --upgrade

You should consider upgrading via the '/Users/qytay/Desktop/data-analysis-code/.venv/bin/python3 -m pip install --upgrade pip' command.


## 2. Configure Databricks Authentication

Set up your Databricks workspace URL and access token. 

**To get your Databricks token:**
1. Go to your Databricks workspace
2. Click on your profile icon (top right)
3. Select "User Settings"
4. Go to "Access Tokens" tab
5. Click "Generate New Token"
6. Copy the generated token

In [43]:
import os
from getpass import getpass

"""
# Configure your Databricks workspace URL and token
# Format: https://<your-instance>.cloud.databricks.com
DATABRICKS_HOST = input("Enter your Databricks workspace URL: ")

# Use getpass to securely input your token
DATABRICKS_TOKEN = getpass("Enter your Databricks access token: ")

# Set environment variables
os.environ['DATABRICKS_HOST'] = DATABRICKS_HOST
os.environ['DATABRICKS_TOKEN'] = DATABRICKS_TOKEN

print("✓ Credentials configured successfully!")

from databricks.sdk import WorkspaceClient

# Initialize workspace client
workspace = WorkspaceClient(
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN
)

print("✓ Workspace connection established!")
print(f"Connected to: {DATABRICKS_HOST}")
"""

'\n# Configure your Databricks workspace URL and token\n# Format: https://<your-instance>.cloud.databricks.com\nDATABRICKS_HOST = input("Enter your Databricks workspace URL: ")\n\n# Use getpass to securely input your token\nDATABRICKS_TOKEN = getpass("Enter your Databricks access token: ")\n\n# Set environment variables\nos.environ[\'DATABRICKS_HOST\'] = DATABRICKS_HOST\nos.environ[\'DATABRICKS_TOKEN\'] = DATABRICKS_TOKEN\n\nprint("✓ Credentials configured successfully!")\n\nfrom databricks.sdk import WorkspaceClient\n\n# Initialize workspace client\nworkspace = WorkspaceClient(\n    host=DATABRICKS_HOST,\n    token=DATABRICKS_TOKEN\n)\n\nprint("✓ Workspace connection established!")\nprint(f"Connected to: {DATABRICKS_HOST}")\n'

## 3. Create a Databricks Workspace Connection

Establish a connection to your Databricks workspace using the configured credentials.

In [55]:
from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()

## 4. Initialize Databricks Client

Test the connection by retrieving your current user information and workspace details.

In [56]:
# Get current user information
current_user = workspace.current_user.me()
print(f"Logged in as: {current_user.user_name}")
print(f"User ID: {current_user.id}")

# List available clusters (first 5)
print("\n----- Available Clusters -----")
try:
    clusters = workspace.clusters.list()
    for i, cluster in enumerate(clusters):
        if i >= 5:
            break
        print(f"  • {cluster.cluster_name} (ID: {cluster.cluster_id})")
        print(f"    State: {cluster.state}")
except Exception as e:
    print(f"  Note: Could not list clusters - {str(e)}")
    print("  This is normal if you don't have cluster access permissions.")

Logged in as: qytay@palo-it.com
User ID: 78030417003700

----- Available Clusters -----
  Note: Could not list clusters - Provided PAT token does not have required scopes: clusters [ReqId: 95e2d5bf-b6b9-46e6-9906-a2b66321ca85]. Config: host=https://dbc-ebe95a09-e87a.cloud.databricks.com, token=***, auth_type=pat. Env: DATABRICKS_HOST, DATABRICKS_TOKEN
  This is normal if you don't have cluster access permissions.


## 5. Create a New Notebook

Programmatically create a new notebook in your Databricks workspace.

In [46]:
from databricks.sdk.service.workspace import ImportFormat, Language
import base64

# Define notebook content
notebook_content = """# My First Databricks Notebook

print("Hello from Databricks!")

# Example: Create a simple DataFrame
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
df = spark.createDataFrame(data, ["Name", "Age"])
df.show()
"""

# Encode content to base64
encoded_content = base64.b64encode(notebook_content.encode()).decode()

# Create notebook in workspace
notebook_path = "/Users/{}/test_notebook".format(current_user.user_name)

try:
    workspace.workspace.import_(
        path=notebook_path,
        format=ImportFormat.SOURCE,
        language=Language.PYTHON,
        content=encoded_content,
        overwrite=True
    )
    print(f"✓ Notebook created successfully at: {notebook_path}")
except Exception as e:
    print(f"Error creating notebook: {str(e)}")

✓ Notebook created successfully at: /Users/qytay@palo-it.com/test_notebook


## 6. Upload Notebook to Workspace

Upload an existing local notebook file to your Databricks workspace.

In [47]:
# Upload local notebook to Databricks workspace
import base64
from databricks.sdk.service.workspace import ImportFormat

def upload_file_to_databricks(local_path, remote_path, importType=ImportFormat.JUPYTER):
    """
    Upload a local notebook file to Databricks workspace
    
    Args:
        local_path: Path to local notebook file
        remote_path: Destination path in Databricks workspace
    """
    try:
        with open(local_path, 'r') as f:
            content = f.read()
        
        encoded_content = base64.b64encode(content.encode()).decode()
        
        workspace.workspace.import_(
            path=remote_path,
            format=importType,
            content=encoded_content,
            overwrite=True
        )
        print(f"✓ Uploaded {local_path} to {remote_path}")
    except Exception as e:
        print(f"Error uploading notebook: {str(e)}")

# Example usage (uncomment to use):
# local_notebook = "01_data_extraction.ipynb"
# remote_notebook = f"/Users/{current_user.user_name}/data_extraction"
# upload_notebook_to_databricks(local_notebook, remote_notebook)

print("Function 'upload_notebook_to_databricks' is ready to use!")

Function 'upload_notebook_to_databricks' is ready to use!


In [48]:
# upload jupyter notebook
upload_file_to_databricks("/Users/qytay/Desktop/data-analysis-code/notebooks/02_data_cleaning.ipynb", "/Users/qytay@palo-it.com/02_data_cleaning.ipynb")

✓ Uploaded /Users/qytay/Desktop/data-analysis-code/notebooks/02_data_cleaning.ipynb to /Users/qytay@palo-it.com/02_data_cleaning.ipynb


## 6. Create Volumes for Data file upload

Create volumes in Unity Catalog to store data files. Volumes are storage locations in Unity Catalog that can be used to store files like CSV, Parquet, images, etc.

In [53]:
from databricks.sdk.service.catalog import VolumeType

def create_databricks_volume(catalog_name, schema_name, volume_name, comment="Data storage volume"):
    """
    Create a volume in Databricks Unity Catalog
    
    Args:
        catalog_name: Name of the catalog
        schema_name: Name of the schema
        volume_name: Name of the volume to create
        comment: Description of the volume
    """
    try:
        # Create the volume
        volume = workspace.volumes.create(
            catalog_name=catalog_name,
            schema_name=schema_name,
            name=volume_name,
            volume_type=VolumeType.MANAGED,
            comment=comment
        )
        print(f"✓ Volume created successfully!")
        print(f"  Catalog: {catalog_name}")
        print(f"  Schema: {schema_name}")
        print(f"  Volume: {volume_name}")
        print(f"  Full path: /Volumes/{catalog_name}/{schema_name}/{volume_name}/")
        return volume
    except Exception as e:
        error_msg = str(e)
        if "ALREADY_EXISTS" in error_msg or "already exists" in error_msg.lower():
            print(f"✓ Volume already exists: /Volumes/{catalog_name}/{schema_name}/{volume_name}/")
        else:
            print(f"✗ Error creating volume: {error_msg}")
        return None

# Example: Create a volume for storing datasets
# Uncomment and modify with your catalog and schema names:
# create_databricks_volume(
#     catalog_name="workspace",
#     schema_name="default", 
#     volume_name="test",
#     comment="Volume for storing uploaded datasets"
# )

print("Function 'create_databricks_volume' is ready to use!")

Function 'create_databricks_volume' is ready to use!


In [54]:
create_databricks_volume(
     catalog_name="workspace",
     schema_name="default", 
     volume_name="test",
     comment="Volume for storing uploaded datasets"
)

✗ Error creating volume: Provided PAT token does not have required scopes: unity-catalog [ReqId: bf9f4878-7202-498d-95b1-b98f3c027ae0]. Config: host=https://dbc-ebe95a09-e87a.cloud.databricks.com, token=***, auth_type=pat. Env: DATABRICKS_HOST, DATABRICKS_TOKEN


## 6.5. Upload Data Files to Volumes in Databricks Catalog

store Data Files into volumes created

In [ ]:
def upload_data_file_to_databricks(local_path, remote_path):
    """
    Upload a data file (CSV, Parquet, etc.) to Databricks workspace
    
    Args:
        local_path: Path to local data file
        remote_path: Destination path in Databricks workspace
    """
    try:
        with open(local_path, 'rb') as f:
            workspace.dbfs.upload(
                path=remote_path,
                src=f,
                overwrite=True
            )
        print(f"✓ Uploaded {local_path} to {remote_path}")
    except Exception as e:
        print(f"✗ Error uploading file: {str(e)}")
        print("\nNote: If uploading to DBFS, use workspace.dbfs.upload() instead")

# Example usage:
local_file = "/Users/qytay/Desktop/data-analysis-code/datasets/weekly-infectious-disease-bulletin-aligned-with-ai-cleaned.csv"
remote_file = "/Volumes/workspace/default/test/datasets/weekly-infectious-disease-bulletin-aligned-with-ai-cleaned.csv"

# Uncomment to upload:
# upload_data_file_to_databricks(local_file, remote_file)

print("Function 'upload_data_file_to_databricks' is ready to use!")

Function 'upload_data_file_to_databricks' is ready to use!


In [ ]:
upload_data_file_to_databricks(local_file, remote_file)

✓ Uploaded /Users/qytay/Desktop/data-analysis-code/datasets/weekly-infectious-disease-bulletin-aligned-with-ai-cleaned.csv to /Volumes/workspace/default/test/datasets/weekly-infectious-disease-bulletin-aligned-with-ai-cleaned.csv


## 7. Connect to Cloud Storage (Azure/AWS/GCP)

Configure connections to cloud storage services. Choose the appropriate section based on your cloud provider.

### AWS S3 Configuration

In [51]:
# AWS S3 Configuration
# Note: This requires proper IAM roles or access keys configured in Databricks

def configure_s3_access(bucket_name, aws_access_key=None, aws_secret_key=None):
    """
    Configure S3 access in Databricks
    
    For production, use IAM roles or instance profiles instead of access keys
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    if aws_access_key and aws_secret_key:
        # Using access keys (not recommended for production)
        spark.conf.set(f"fs.s3a.access.key", aws_access_key)
        spark.conf.set(f"fs.s3a.secret.key", aws_secret_key)
    
    print(f"✓ S3 access configured for bucket: {bucket_name}")
    return f"s3a://{bucket_name}/"

# Example usage:
# s3_path = configure_s3_access("my-bucket-name")
# df = spark.read.csv(s3_path + "data.csv", header=True)

print("AWS S3 configuration function ready!")

AWS S3 configuration function ready!


In [52]:
configure_s3_access(bucket_name, aws_access_key=None, aws_secret_key=None)

NameError: name 'bucket_name' is not defined

### Azure Blob Storage Configuration

In [ ]:
# Azure Blob Storage Configuration
def configure_azure_blob(storage_account, container_name, account_key=None):
    """
    Configure Azure Blob Storage access in Databricks
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    if account_key:
        spark.conf.set(
            f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
            account_key
        )
    
    print(f"✓ Azure Blob access configured")
    print(f"  Storage Account: {storage_account}")
    print(f"  Container: {container_name}")
    
    return f"wasbs://{container_name}@{storage_account}.blob.core.windows.net/"

# Example usage:
# azure_path = configure_azure_blob("mystorageaccount", "mycontainer")
# df = spark.read.parquet(azure_path + "data/")

print("Azure Blob configuration function ready!")

### Google Cloud Storage (GCS) Configuration

In [ ]:
# Google Cloud Storage Configuration
def configure_gcs(bucket_name, service_account_json_path=None):
    """
    Configure GCS access in Databricks
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    if service_account_json_path:
        spark.conf.set(
            "google.cloud.auth.service.account.json.keyfile",
            service_account_json_path
        )
    
    print(f"✓ GCS access configured for bucket: {bucket_name}")
    return f"gs://{bucket_name}/"

# Example usage:
# gcs_path = configure_gcs("my-gcs-bucket")
# df = spark.read.json(gcs_path + "data.json")

print("GCS configuration function ready!")

## 8. Test Cloud Connection

Verify your cloud storage connection with basic operations.

In [ ]:
# Test cloud storage connection
def test_cloud_storage(path):
    """
    Test cloud storage connectivity by listing files
    
    Args:
        path: Cloud storage path (s3a://, wasbs://, or gs://)
    """
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        # List files in the path
        files = dbutils.fs.ls(path)
        
        print(f"✓ Successfully connected to: {path}")
        print(f"\nFound {len(files)} items:")
        for file in files[:10]:  # Show first 10 items
            print(f"  • {file.name} ({'DIR' if file.isDir() else file.size} bytes)")
        
        if len(files) > 10:
            print(f"  ... and {len(files) - 10} more items")
            
        return True
    except Exception as e:
        print(f"✗ Connection test failed: {str(e)}")
        return False

# Example usage (uncomment and modify with your actual path):
# test_cloud_storage("s3a://my-bucket/data/")
# test_cloud_storage("wasbs://container@account.blob.core.windows.net/")
# test_cloud_storage("gs://my-gcs-bucket/data/")

print("\nTest function ready! Use test_cloud_storage(path) to verify connectivity.")

## Next Steps

You've successfully set up Databricks cloud connectivity! Here are some next steps:

1. **Create a cluster** in your Databricks workspace to run computations
2. **Upload your data** to cloud storage (S3, Azure Blob, or GCS)
3. **Create notebooks** to analyze your data using Spark
4. **Set up jobs** to automate your data pipelines
5. **Configure Unity Catalog** for data governance

### Useful Resources:
- [Databricks Documentation](https://docs.databricks.com/)
- [Databricks SDK for Python](https://databricks-sdk-py.readthedocs.io/)
- [Cloud Storage Integration](https://docs.databricks.com/data/data-sources/index.html)
- [Databricks CLI](https://docs.databricks.com/dev-tools/cli/index.html)